<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.9}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Alumno:} & Jiménez Rivera Emiliano Kaleb \\[6pt]
\textbf{Fecha de realización:} & 25/09/2026
\end{array}

</center>

# 1. Base de Conocimiento (Prolog)

La base de conocimiento define la ubicación ideal de los productos y las reglas para el diagnóstico y la planificación

# 2. Estados Iniciales y Finales

Para un programa de toma de decisiones, definimos los estados de la siguiente manera:Estado Inicial: El robot se encuentra en la "posición inicial" (centro del triángulo que forman los estantes). Los estantes tienen una distribución de productos que puede o no coincidir con la ideal.Estado Final:
- 1.  El cliente ha recibido el producto solicitado (ej. "Aquí está el refresco").
- 2.  Todos los productos observados durante la tarea han sido reubicados en sus estantes correctos según la base de conocimiento ("Productos Acomodados").

# 3. Integración en Python (Uso de pyswip)
Para conectar esto con Python, se utiliza la librería pyswip. El flujo consiste en que Python maneja la interfaz y el robot, mientras Prolog toma las decisiones lógicas.

In [ ]:
# 1. Instalar el motor de SWI-Prolog en el sistema
!apt-get install -y swi-prolog

# 2. Instalar la librería puente para Python
!pip install pyswip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swi-prolog is already the newest version (9.0.4+dfsg-3.1ubuntu4).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


Crear un archivo como base de conocimiento


% Ubicaciones ideales (Creencia inicial del sistema)

ideal(estante_bebidas, [cerveza, refresco]).

ideal(estante_comida, [sopa, cereal]).

ideal(estante_pan, [galletas]).


% Estado actual (Hechos dinámicos que Python actualizará tras la observación)

% Ejemplo de un estado con desorden (Caso 1.2 b)

ubicado_en(estante_bebidas, refresco).

ubicado_en(estante_bebidas, cerveza).

ubicado_en(estante_bebidas, sopa). % Producto desacomodado

ubicado_en(estante_comida, cereal).

ubicado_en(estante_pan, galletas).

% Regla de Diagnóstico: Identifica qué productos no están en su lugar

diagnostico(Producto, EstanteActual, EstanteCorrecto) :-

    ubicado_en(EstanteActual, Producto),

    ideal(EstanteCorrecto, ProductosIdeales),

    member(Producto, ProductosIdeales),

    EstanteActual \= EstanteCorrecto.


% Regla de Planificación: Genera las acciones necesarias

plan(entregar(P)) :- ubicado_en(_, P).

plan(acomodar(P, Destino)) :- diagnostico(P, _, Destino).

In [ ]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Carga la base de conocimiento

def ejecutar_asistente(producto_buscado):
    print(f"Buscando: {producto_buscado}")

    # 1. Consultar a Prolog por el plan de entrega
    plan_entrega = list(prolog.query(f"plan(entregar({producto_buscado}))"))

    # 2. Consultar si hay algo que reacomodar (Diagnóstico)
    objetos_desacomodados = list(prolog.query("plan(acomodar(P, Destino))"))

    if objetos_desacomodados:
        for obj in objetos_desacomodados:
            print(f"Acción: Mover {obj['P']} al {obj['Destino']}")

    if plan_entrega:
        print(f"Acción: Entregar {producto_buscado} al cliente.")
    else:
        print("Error: Producto no encontrado en el sistema.")

ejecutar_asistente("refresco")

Buscando: refresco
Acción: Mover sopa al estante_comida
Acción: Entregar refresco al cliente.


# Escenario del Desafío:

El cliente solicita un refresco. El robot llega al "Estante de Bebidas" y lo encuentra vacío. Al inspeccionar el "Estante de Comida", descubre que allí están la cerveza, la sopa y las galletas, mientras que el refresco y el cereal están en el "Estante de Pan".
**Reto para el programador:**
Implementar una regla en Prolog que permita al robot priorizar: ¿Debe acomodar todo primero o entregar el refresco cuanto antes para satisfacer al cliente?.

Manejar el "Error de Búsqueda": Si el robot va a un estante esperando algo y no lo encuentra, debe disparar un nuevo diagnóstico (re-planificación en tiempo real).

Restricción: El robot solo puede cargar un máximo de dos objetos a la vez. ¿Cómo optimiza el movimiento entre los 3 estantes para minimizar la distancia recorrida?

In [ ]:
prolog.consult("supermercado_desafio.pl")

mundo = {"estante_bebidas": [],
         "estante_comida": ["cerveza", "sopa", "galletas"],
         "estante_pan": ["refresco", "cereal"]}

def consulta(q, var):
  """devuelve una lista de valores de una variable en una consulta"""
  return [str(r[var]) for r in prolog.query(q)]


def cierto(q):
  """Nos indica si la consulta tiene al menos una solucion"""
  return bool(list(prolog.query(q)))

def observar(estante):
  """actualiza creencias del robot con lo que ve en el estante"""
  prolog.retractall(f"ubicado_en({estante}, _)")
  for p in mundo[estante]:
    prolog.assertz(f"ubicado_en({estante}, {p})")

def desacomodados(estante):
  return consulta(f"prioridad(2, acomodar(P, {estante}, _))", "P")


def proceso(producto):
  prolog.retractall("ubicado_en(_, _)")
  prolog.assertz(f"solicitado({producto})")
  cap = int(consulta("capacidad(C)", "C")[0])
  movimientos = 0

  #primero busca donde deberia estar
  esperado = consulta(f"va_en({producto}, E)", "E")[0]
  for estante in [esperado] + [e for e in mundo if e != esperado]:
    movimientos += 1
    observar(estante)
    if cierto(f"prioridad(1, entregar({producto}))"):
      print(f"{producto} encontrado en {estante}")
      break
    print(f"Error de búsqueda en {estante}. Desacomodados: {desacomodados(estante)}")


  #primera prioridad seria  entregar y llevando de paso un objeto desacomodado
  carga = [producto] + desacomodados(estante)[:cap - 1]
  for p in carga:
    prolog.retractall(f"ubicado_en(_, {p})")
  movimientos += 1
  print(f"Carga {carga} -> centro. Entregar {producto} al ciente")
  carga.remove(producto)
  prolog.retractall("solicitado(_)")


  #segunda prioridad acomodar de dos en dos yendo al destino de lo que se quiere
  #entregar
  while carga or cierto("prioridad(2, acomodar(_, _, _))"):
    if carga:
      estante = consulta(f"va_en({carga[0]}, E)", "E")[0]
    else:
      estante = consulta("prioridad(2, acomodar(_, A, _))", "A")[0]
    movimientos += 1
    for p in [p for p in carga if cierto(f"va_en({p}, {estante})")]:
      carga.remove(p)
      prolog.assertz(f"ubicado_en({estante}, {p})")
    for p in desacomodados(estante)[:cap - len(carga)]:
      carga.append(p)
      prolog.retractall(f"ubicado_en({estante}, {p})")
    print(f"Mover a {estante}. Ahora carga: {carga}")
  print(f"productos acomodados. Movimientos totales: {movimientos}")

proceso("refresco")


Error de búsqueda en estante_bebidas. Desacomodados: []
Error de búsqueda en estante_comida. Desacomodados: ['cerveza', 'galletas']
refresco encontrado en estante_pan
Carga ['refresco', 'cereal'] -> centro. Entregar refresco al ciente
Mover a estante_comida. Ahora carga: ['cerveza', 'galletas']
Mover a estante_bebidas. Ahora carga: ['galletas']
Mover a estante_pan. Ahora carga: []
productos acomodados. Movimientos totales: 7


Algunas notas sobre la implementación:

La priorización se puso como nivel 1 poder entregar el producto solicitado y en el nivel 2 acomodar todo lo demás. Es por eso que el robot atiende primero al cliente. Sin embarog, esto no le impide ir acomodando cosas en el camino si tiene espaicos de carga disponibles. Podemos ver que por ejemplo que el robot salió del estante de pan con el refresco y el cereal, entregó el refresco en el centro y siguió directo al estante de comida a dejar el cereal donde iba.

Los errores de búsqueda, estos suceden cuando el robot va a buscar un producto y no está donde debería de estar, el robot actualiza sus creencias de manera instantánea con retractall y assertz y vuelve a consultar antes de ir al estante.

#Conclusiones finales del estudiante

En esta práctica pudimos consturir un agente lógica basado en conocimiento dentro de un archivo de prolog para un robot de supermercado. Prolog es el encargado de la lógica: i.e. guarda la ubicación ideal de cada producto, también diagnostica y le dice al robot qué está acomodado en qué estanterías y define las prioridades para el robot. Ya mediante python es que podemos simular al robot y la comunicación con el conocimiento se hace mediante pyswip. Usamos en prolog la declaración dyanmic que es primoridal para poner hechos que cambian con el tiempo y con las acciones y modificaciones en el conocimiento y/o entorno. Aquí se evidenció cómo es que un robot puede partir de creencias que están equivocadas, pero que va ajustando conforme aprender de su entorno; además de que el robot hizo re-planificaciones dos veces antes de encontrar el refresco, y aun así pudo compeltar la entrega y dejó todo acomodado en tal solo 7 movimientos respetando la condición de límite de carga. Por tanto, en un programa es fundamental poder escribir reglas declarativas del control de un agente, ya que nos permite cambiar su comportamiento (mediante prioridades) e ir modificando su conocimiento. Quiero suponer que esta es una introducción a lo que se podrían transformar en sistemas expertos, que son una rama de todo lo que tiene que ver automatizaciones, y en general me gusta cómo se están abordando los temas.

<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>